In [70]:
from pulp import *
import pandas as pd

# masses in dalton
phosphorus = 30.973761998
oxygen = 31.989829239
d_ribose = 150.05282342

tail_5prime = [d_ribose, oxygen, 2 * oxygen + phosphorus, oxygen]

base_masses = pd.read_csv("../masses_all.tsv", sep="\t").set_index(
    "nucleoside", drop=False
)
# dbg:
# masses = masses.loc[["A", "C", "G", "U"]]

In [71]:
def get_representatives(base_masses):
    # collapse nucleosides that have the same mass
    return (
        base_masses.groupby("monoisotopic_mass")
        .apply(lambda x: x.iloc[0])
        .set_index("nucleoside", drop=False)
    )


base_masses_representatives = get_representatives(base_masses)
base_masses_representatives

/tmp/ipykernel_1694558/1235673937.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[0])


,nucleoside,monoisotopic_mass
nucleoside,,
C,C,243.08552
U,U,244.06954
A,A,267.09675
1A,1A,281.11240
19A,19A,282.09640
G,G,283.09167
67A,67A,295.09170
01A,01A,295.12810
019A,019A,296.11210


In [72]:
base_masses

,nucleoside,monoisotopic_mass
nucleoside,,
A,A,267.09675
G,G,283.09167
C,C,243.08552
U,U,244.06954
1A,1A,281.11240
...,...,...
10G,10G,409.15970
102G,102G,425.15470
105G,105G,538.20230


In [73]:
import random, re
from scipy.stats import norm

nucleoside_re = re.compile(r"\d*[ACGU]")

true_sequence = nucleoside_re.findall("CCUGUU")
print(true_sequence)
n_fragments = 60

# sample random fragments from true sequence
fragment_noise = norm.rvs(scale=0.5, size=n_fragments)


def random_fragment():
    l = random.randint(0, len(true_sequence) - 1)
    r = random.randint(l + 1, len(true_sequence))
    return l, r


fragments = [random_fragment() for _ in range(n_fragments)]

true_fragment_masses = [
    sum(base_masses.loc[b, "monoisotopic_mass"] for b in true_sequence[l:r])
    for l, r in fragments
]

fragment_masses = [
    max(true_mass + noise, 0.0)
    for noise, true_mass in zip(fragment_noise, true_fragment_masses)
]

start_fragments = [i for i in range(n_fragments) if fragments[i][0] == 0]
end_fragments = [i for i in range(n_fragments) if fragments[i][1] == len(true_sequence)]

fragments, fragment_masses, fragment_noise

['C', 'C', 'U', 'G', 'U', 'U']


([(4, 5),
  (2, 3),
  (0, 3),
  (3, 6),
  (0, 6),
  (3, 4),
  (5, 6),
  (0, 6),
  (5, 6),
  (1, 2),
  (3, 6),
  (1, 4),
  (5, 6),
  (2, 6),
  (5, 6),
  (5, 6),
  (2, 6),
  (3, 5),
  (4, 5),
  (5, 6),
  (3, 6),
  (2, 4),
  (0, 5),
  (5, 6),
  (1, 2),
  (2, 6),
  (0, 2),
  (1, 5),
  (4, 6),
  (5, 6),
  (3, 6),
  (4, 6),
  (2, 3),
  (4, 6),
  (5, 6),
  (5, 6),
  (0, 2),
  (1, 2),
  (5, 6),
  (3, 4),
  (3, 4),
  (3, 6),
  (1, 5),
  (1, 5),
  (0, 3),
  (1, 4),
  (5, 6),
  (5, 6),
  (5, 6),
  (3, 4),
  (2, 4),
  (3, 6),
  (3, 6),
  (2, 6),
  (1, 3),
  (3, 6),
  (0, 1),
  (4, 6),
  (3, 5),
  (1, 4)],
 [np.float64(244.2097300065327),
  np.float64(244.61824878182844),
  np.float64(730.1634782480007),
  np.float64(770.9350939993342),
  np.float64(1501.7960336314527),
  np.float64(283.46977110352964),
  np.float64(243.66218487166677),
  np.float64(1500.8362323241283),
  np.float64(244.61957372782976),
  np.float64(242.6104277366917),
  np.float64(771.5402250121628),
  np.float64(770.050189002267)

In [74]:
from itertools import combinations


def reduce_alphabet_by_fragments(the_base_masses):
    # TODO: the -2.0 should be informed by the variance in the measurements
    min_plausible_nucleoside_diff = the_base_masses["monoisotopic_mass"].min() - 2.0

    start_masses = pd.Series(sorted(fragment_masses[i] for i in start_fragments))
    end_masses = pd.Series(sorted(fragment_masses[i] for i in end_fragments))

    print(start_masses)
    print(end_masses)

    def mass_diffs(masses):
        diffs = set(masses[1:].values - masses[:-1].values)
        print(diffs)
        return diffs

    diffs = sorted(mass_diffs(start_masses) | mass_diffs(end_masses))

    def is_similar(mass_a, mass_b):
        return abs(mass_a - mass_b) < 1.0

    def explain_diffs(diffs):
        explanations = {
            diff: [
                item.nucleoside
                for item in the_base_masses.itertuples()
                if is_similar(diff, item.monoisotopic_mass)
            ]
            for diff in diffs
        }
        explanations.update(
            {
                diff: [
                    (item_a.nucleoside, item_b.nucleoside)
                    for item_a, item_b in combinations(the_base_masses.itertuples(), 2)
                    if is_similar(
                        diff, item_a.monoisotopic_mass + item_b.monoisotopic_mass
                    )
                ]
                for diff in diffs
                if not explanations[diff]
            }
        )
        return explanations

    def get_similar_nucleosides():
        i_diff = 0
        sorted_masses = the_base_masses.sort_values("monoisotopic_mass")
        for item in sorted_masses.itertuples():
            # move until close enough to next mass in base_masses
            # (diffs are sorted)
            # TODO the -1.0 should be informed by the variance in the measurements
            while i_diff < len(diffs) and diffs[i_diff] < item.monoisotopic_mass - 1.0:
                i_diff += 1
            j = 0
            # TODO the +1.0 should be informed by the variance in the measurements
            while (
                i_diff + j < len(diffs)
                and diffs[i_diff + j] <= item.monoisotopic_mass + 1.0
            ):
                diff_mass = diffs[i_diff + j]
                if is_similar(diff_mass, item.monoisotopic_mass):
                    yield item.nucleoside
                j += 1

    explanations = explain_diffs(diffs)

    # unexplained differences, only consider those
    unexplained = [
        diff
        for diff, expls in explanations.items()
        if not expls and diff > min_plausible_nucleoside_diff
    ]

    if unexplained:
        # at least one fragment differs by more than two nucleosides,
        # so we can't reduce the alphabet with this heuristic so far
        print("Unexplained differences:", unexplained)
        return None

    def get_nucleosides(explanations):
        for expls in explanations.values():
            for expl in expls:
                if isinstance(expl, tuple):
                    yield from expl
                else:
                    yield expl

    all_nucleosides = list(set(get_nucleosides(explanations)))

    print(all_nucleosides)

    return the_base_masses.loc[all_nucleosides]

In [75]:
reduced_alphabet_masses = reduce_alphabet_by_fragments(base_masses_representatives)
if reduced_alphabet_masses is None:
    reduced_alphabet_masses = base_masses_representatives

0     242.708747
1     485.518026
2     486.030736
3     730.163478
4     730.210258
5    1257.278162
6    1500.836232
7    1501.796034
dtype: float64
0      242.789308
1      243.293808
2      243.338969
3      243.490029
4      243.662185
5      243.769847
6      243.799190
7      243.961454
8      244.118486
9      244.243378
10     244.301700
11     244.619574
12     244.624400
13     244.700388
14     488.194351
15     488.352164
16     488.455500
17     488.760197
18     770.867246
19     770.935094
20     771.399296
21     771.518659
22     771.540225
23     771.743325
24     771.828289
25     772.075585
26    1013.982209
27    1014.793200
28    1015.399298
29    1015.477968
30    1500.836232
31    1501.796034
dtype: float64
{np.float64(0.5127099274389479), np.float64(0.04677929989452423), np.float64(0.9598013073243692), np.float64(527.0679045118536), np.float64(242.80927958890004), np.float64(243.55807026437947), np.float64(244.13274207055917)}
{np.float64(0.5044994915095629), 

In [76]:
def estimate_seq(base_masses):
    prob = LpProblem("RNA sequencing", LpMinimize)
    # i = 1,...,n: positions in the sequence
    # j = 1,...,m: fragments
    # b = 1,...,k: (modified) bases

    n = len(true_sequence)
    m = len(fragment_masses)
    bases = base_masses.index.values

    # x: binary variables indicating fragment j presence at position i
    x = [
        [
            LpVariable(f"x_{i},{j}", lowBound=0, upBound=1, cat=LpInteger)
            for j in range(m)
        ]
        for i in range(n)
    ]
    # y: binary variables indicating base at position i
    y = [
        {
            b: LpVariable(f"y_{i},{b}", lowBound=0, upBound=1, cat=LpInteger)
            for b in bases
        }
        for i in range(n)
    ]
    # z: binary variables indicating product of x and y
    z = [
        [
            {
                b: LpVariable(f"z_{i},{j},{b}", lowBound=0, upBound=1, cat=LpInteger)
                for b in bases
            }
            for j in range(m)
        ]
        for i in range(n)
    ]
    # weight_diff_abs: absolute value of weight_diff
    predicted_mass_diff_abs = [
        LpVariable(f"predicted_mass_diff_abs_{j}", lowBound=0, cat=LpContinuous)
        for j in range(m)
    ]
    # weight_diff: difference between fragment monoisotopic mass and sum of masses of bases in fragment as estimated in the MILP
    preficted_mass_diff = [
        fragment_masses[j]
        - lpSum(
            [
                z[i][j][b] * base_masses.loc[b, "monoisotopic_mass"]
                for i in range(n)
                for b in bases
            ]
        )
        for j in range(m)
    ]

    # optimization function
    prob += lpSum([predicted_mass_diff_abs[j] for j in range(m)])

    # select one base per position
    for i in range(n):

        prob += lpSum([y[i][b] for b in bases]) == 1

    # fill z with the product of binary variables x and y
    for i in range(n):
        for j in range(m):
            for b in bases:
                prob += z[i][j][b] <= x[i][j]
                prob += z[i][j][b] <= y[i][b]
                prob += z[i][j][b] >= x[i][j] + y[i][b] - 1

    # ensure that fragment is aligned continuously
    # (no gaps: if x[i1,j] = 1 and x[i2,j] = 1, then x[i_between,j] = 1)
    for j in range(m):
        for i1, i2 in combinations(range(n), 2):
            # i2 and i1 are inclusive
            assert i2 > i1
            if i2 - i1 > 1:
                prob += (x[i1][j] + x[i2][j] - 1) * (i2 - i1 - 1) <= lpSum(
                    [x[i_between][j] for i_between in range(i1 + 1, i2)]
                )
                # for i_between in range(i1 + 1, i2):
                #     prob += x[i1][j] + x[i2][j] - 1 <= x[i_between][j]

    # ensure that start fragments are aligned at the beginning of the sequence
    for j in start_fragments:
        x[0][j].setInitialValue(1)
        x[0][j].fixValue()

    # ensure that end fragments are aligned at the end of the sequence
    for j in end_fragments:
        x[n - 1][j].setInitialValue(1)
        x[n - 1][j].fixValue()

    # constrain weight_diff_abs to be the absolute value of weight_diff
    for j in range(m):
        prob += predicted_mass_diff_abs[j] >= preficted_mass_diff[j]
        prob += predicted_mass_diff_abs[j] >= -preficted_mass_diff[j]

    import pulp

    gurobi = pulp.getSolver("GUROBI_CMD")
    gurobi.msg = False
    result_ = prob.solve(gurobi)

    def get_base(i):
        for b in bases:
            if y[i][b].value() == 1:
                return b
        return None

    return [get_base(i) for i in range(n)], x, y, z, preficted_mass_diff

In [77]:
from collections import defaultdict
from sklearn.cluster import KMeans
import numpy as np

reduce_to_k = 6


def reduce_alphabet(masses):
    kmeans = KMeans(n_clusters=reduce_to_k, random_state=213126)
    kmeans.fit(masses["monoisotopic_mass"].values.reshape(-1, 1))
    pseudo_nucs = [f"n{k}" for k in range(reduce_to_k)]
    nuc_mapping = defaultdict(list)
    for nuc, label in zip(masses["nucleoside"], kmeans.labels_):
        nuc_mapping[pseudo_nucs[label]].append(nuc)
    return (
        pd.DataFrame(
            {
                "nucleoside": pseudo_nucs,
                "monoisotopic_mass": kmeans.cluster_centers_[:, 0],
            }
        ).set_index("nucleoside", drop=False),
        nuc_mapping,
    )

In [78]:
def expand_alphabet(seq, nuc_mapping):
    return {nuc for pseudo_nuc in seq for nuc in nuc_mapping[pseudo_nuc]}

In [79]:
start_fragments, end_fragments

([2, 4, 7, 22, 26, 36, 44, 56],
 [3,
  4,
  6,
  7,
  8,
  10,
  12,
  13,
  14,
  15,
  16,
  19,
  20,
  23,
  25,
  28,
  29,
  30,
  31,
  33,
  34,
  35,
  38,
  41,
  46,
  47,
  48,
  51,
  52,
  53,
  55,
  57])

In [94]:
current_masses = reduced_alphabet_masses
predicted_sequence = None
last_masses_count = None

while True:
    print(f"remaining masses: {current_masses}")

    if (
        current_masses.shape[0] > reduce_to_k
        and current_masses.shape[0] != last_masses_count
    ):
        pseudo_masses, nuc_mapping = reduce_alphabet(current_masses)
        predicted_sequence, _, _, _, _ = estimate_seq(pseudo_masses)
        print(predicted_sequence, nuc_mapping)
    else:
        predicted_sequence, x, y, z, predicted_mass_diff = estimate_seq(current_masses)
        break

    last_masses_count = current_masses.shape[0]
    current_masses = current_masses.loc[list(expand_alphabet(predicted_sequence, nuc_mapping))]
predicted_sequence

remaining masses:            nucleoside  monoisotopic_mass
nucleoside                              
C                   C          243.08552
U                   U          244.06954
A                   A          267.09675
1A                 1A          281.11240
19A               19A          282.09640
G                   G          283.09167
67A               67A          295.09170
01A               01A          295.12810
019A             019A          296.11210
68A               68A          297.10730
7G                 7G          299.12300
100G             100G          307.09170
64A               64A          309.10730
066A             066A          309.14370
01G               01G          311.12300
27G               27G          313.13860
4G                 4G          321.10730
103G             103G          324.11820
022G             022G          325.13860
621A             621A          327.10010
027G             027G          327.15430
42G               42G          335.1230

/home/johannes/micromamba/envs/pystats/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


['n4', 'n4', 'n4', 'n4', 'n4', 'n4'] defaultdict(<class 'list'>, {'n4': ['C', 'U', 'A', '1A', '19A', 'G'], 'n1': ['67A', '01A', '019A', '68A', '7G', '100G', '64A', '066A', '01G', '27G', '4G', '103G', '022G', '621A', '027G', '42G', '61A', '342G', '60A'], 'n0': ['65A', '2161A', '69A', '2160A'], 'n5': ['10G', '62A', '47G', '102G', '63A', '2165A', '347G', '2164A'], 'n3': ['348G', '3470G', '2162A', '3480G', '2163A', '00A', '00G'], 'n2': ['3483G', '34830G', '105G', '34832G', '104G']})
remaining masses:            nucleoside  monoisotopic_mass
nucleoside                              
A                   A          267.09675
1A                 1A          281.11240
G                   G          283.09167
C                   C          243.08552
19A               19A          282.09640
U                   U          244.06954


/home/johannes/micromamba/envs/pystats/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


['C', 'C', 'U', 'G', 'U', 'U']

In [95]:
import altair as alt
import polars as pl

n = len(true_sequence)
m = len(fragment_masses)
fragment_coverage = pl.from_dicts(
    [
        {
            "l": min(i for i in range(n) if x[i][j].value() == 1) - 0.5,
            "r": max(i for i in range(n) if x[i][j].value() == 1) + 0.5,
            "j": j,
            "mass_info": f"{fragment_masses[j]:.2f} ({predicted_mass_diff[j].value():.2g})",
            "type": "predicted",
        }
        for j in range(m)
    ]
)

true_fragments = (
    pl.from_records(fragments)
    .transpose()
    .rename({"column_0": "l", "column_1": "r"})
    .with_columns(pl.col("r") - 1)
).with_columns(
    pl.col("l") - 0.5,
    pl.col("r") + 0.5,
    pl.arange(start=0, end=m).alias("j"),
    pl.Series(
        name="mass_info", values=[f"{mass:.2f}" for mass in true_fragment_masses]
    ),
    pl.lit("truth").alias("type"),
)
data = pl.concat([fragment_coverage, true_fragments])

In [91]:
data

l,r,j,mass_info,type
f64,f64,i64,str,str
4.5,5.5,0,"""244.21 (0.14)""","""predicted"""
4.5,5.5,1,"""244.62 (0.55)""","""predicted"""
-0.5,2.5,2,"""730.16 (-0.077)""","""predicted"""
2.5,5.5,3,"""770.94 (-0.3)""","""predicted"""
-0.5,5.5,4,"""1501.80 (0.32)""","""predicted"""
…,…,…,…,…
2.5,5.5,55,"""771.23""","""truth"""
-0.5,0.5,56,"""243.09""","""truth"""
3.5,5.5,57,"""488.14""","""truth"""


In [92]:
fragment_coverage

l,r,j,mass_info,type
f64,f64,i64,str,str
4.5,5.5,0,"""244.21 (0.14)""","""predicted"""
4.5,5.5,1,"""244.62 (0.55)""","""predicted"""
-0.5,2.5,2,"""730.16 (-0.077)""","""predicted"""
2.5,5.5,3,"""770.94 (-0.3)""","""predicted"""
-0.5,5.5,4,"""1501.80 (0.32)""","""predicted"""
…,…,…,…,…
2.5,5.5,55,"""771.40 (0.17)""","""predicted"""
-0.5,0.5,56,"""242.71 (-0.38)""","""predicted"""
3.5,5.5,57,"""488.46 (0.32)""","""predicted"""


In [104]:
seq_data = pl.DataFrame({
    "nucleoside": true_sequence + predicted_sequence,
    "pos": list(range(len(true_sequence))) * 2,
    "type": ["truth"] * len(true_sequence) + ["predicted"] * len(predicted_sequence),
})

base = alt.Chart(data)
((
    base.mark_rule().encode(
        alt.X("l").axis(title=None, labels=False, ticks=False),
        alt.X2("r"),
        alt.Y("type"),
    )
    + base.mark_text(align="left", dx=3).encode(
        alt.X("r").axis(title=None, labels=False, ticks=False),
        alt.Y("type"),
        alt.Text("mass_info"),
    )
).facet(row="j") & alt.Chart(seq_data).mark_text().encode(
    alt.X("pos").title(None),
    alt.Y("type"),
    alt.Text("nucleoside"),
    alt.Color("nucleoside", scale=alt.Scale(scheme="category20")).legend(None),
)).resolve_scale(x="shared")

alt.VConcatChart(...)

In [85]:
z[-1][-1]

{'A': z_5,59,A,
 '1A': z_5,59,1A,
 'G': z_5,59,G,
 'C': z_5,59,C,
 '19A': z_5,59,19A,
 'U': z_5,59,U}

In [86]:
[[z[i][-2][k].value() for k in current_masses["nucleoside"]] for i in range(n)]

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]

In [87]:
[x[i][-2].value() for i in range(n)]

[0.0, 0.0, 1.0, 1.0, 0.0, 0.0]

In [88]:
fragment_coverage

l,r,j,mass_info,type
f64,f64,i64,str,str
4.5,5.5,0,"""244.21 (0.14)""","""predicted"""
4.5,5.5,1,"""244.62 (0.55)""","""predicted"""
-0.5,2.5,2,"""730.16 (-0.077)""","""predicted"""
2.5,5.5,3,"""770.94 (-0.3)""","""predicted"""
-0.5,5.5,4,"""1501.80 (0.32)""","""predicted"""
…,…,…,…,…
2.5,5.5,55,"""771.40 (0.17)""","""predicted"""
-0.5,0.5,56,"""242.71 (-0.38)""","""predicted"""
3.5,5.5,57,"""488.46 (0.32)""","""predicted"""
